# Medición de velocidad

Mide cuánto tarda una generación del modelo real, para decidir cuántos arranques y qué subcatálogo caben.

**Uso:** elija una GPU T4 y ejecute todas las celdas. La primera vez el entorno se instala y se reinicia; después, ejecute todas las celdas otra vez.

In [ ]:
import os

MARCA = "/content/.entorno_listo"
REPO = "https://github.com/Seloga26/ADA2026_2"

if not os.path.exists(MARCA):
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q {REPO} /content/ADA2026_2

    import io
    import urllib.request
    import zipfile

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    PAQUETES = [
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

    open(MARCA, "w").close()
    print("Entorno instalado. El entorno se reinicia: vuelva a ejecutar todas las celdas.")
    os.kill(os.getpid(), 9)

%cd /content/ADA2026_2
print("Entorno listo")

In [ ]:
import json
import random
import time

import torch

from ayudas import cargar_datos, cargar_modelo, dividir
from busqueda_prompt.contado import OraculoContado
from busqueda_prompt.contratos import OPCIONES_POR_RANURA, RANURAS
from oraculo import Oraculo

modelo = cargar_modelo("qwen17b")
busqueda, validacion = dividir(cargar_datos())

# Sin caché: cada generación se mide de verdad.
real = Oraculo(modelo, busqueda, validacion, cache=None, lote=16, presupuesto=32_000_000)
oraculo = OraculoContado(real)

# Configuración al azar con semilla fija: solo mide velocidad, no busca nada.
rng = random.Random(1)
config = {r: rng.randrange(OPCIONES_POR_RANURA) for r in RANURAS} | {"temperatura": 0.0}

torch.cuda.reset_peak_memory_stats()
inicio = time.perf_counter()
r = oraculo.evaluar(config, busqueda, semilla=1)
segundos = time.perf_counter() - inicio

por_generacion = segundos / oraculo.consultas
medicion = {
    "modelo": modelo.nombre,
    "gpu": torch.cuda.get_device_name(0),
    "config": config,
    "generaciones": oraculo.consultas,
    "segundos": round(segundos, 2),
    "segundos_por_generacion": round(por_generacion, 3),
    "vram_pico_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "precision": r.precision,
}
print(json.dumps(medicion, indent=1, ensure_ascii=False))

for nombre, generaciones in (
    ("una pasada de coordenadas", 4_000),
    ("subcatálogo de 32 configuraciones", 3_200),
    ("subcatálogo de 243 configuraciones", 24_300),
):
    print(f"{nombre}: {generaciones * por_generacion / 3600:.1f} h como máximo")

with open("velocidad.json", "w", encoding="utf-8") as archivo:
    json.dump(medicion, archivo, indent=1, ensure_ascii=False)

In [ ]:
from google.colab import files

files.download("velocidad.json")